# Naive Bayes Classifier — From Scratch on the Play Tennis Dataset

In this notebook we build a **Categorical Naive Bayes classifier completely from scratch** — no scikit-learn for the core algorithm — and apply it to the classic *Play Tennis* dataset.

By the end you will:
- Understand every formula the classifier uses
- Code each formula step-by-step
- Make predictions and evaluate accuracy
- Verify our implementation against scikit-learn's `CategoricalNB`

---

## The Dataset

| Column | Values | Role |
|---|---|---|
| `outlook` | Sunny, Overcast, Rain | Feature |
| `temp` | Hot, Mild, Cool | Feature |
| `humidity` | High, Normal | Feature |
| `wind` | Weak, Strong | Feature |
| `play` | Yes, No | **Target** |

The question the classifier answers: **given today's weather conditions, should we play tennis?**

---

## Why Categorical Naive Bayes?

All four features are **finite, unordered categories** — there is no natural numerical distance between *Sunny* and *Rain*. Gaussian NB assumes continuous features; Multinomial NB counts word occurrences. **Categorical NB** is designed exactly for this: each feature gets its own categorical distribution, estimated separately per class.

---

## Step 0 — Import Libraries and Load Data

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv('datasets/play_tennis.csv')
df

,day,outlook,temp,humidity,wind,play
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes
5,D6,Rain,Cool,Normal,Strong,No
6,D7,Overcast,Cool,Normal,Strong,Yes
7,D8,Sunny,Mild,High,Weak,No
8,D9,Sunny,Cool,Normal,Weak,Yes
9,D10,Rain,Mild,Normal,Weak,Yes


In [23]:
print('Shape:', df.shape)
print('\nClass distribution:')
print(df['play'].value_counts())
print('\nFeature unique values:')
for col in ['outlook', 'temp', 'humidity', 'wind']:
    print(f'  {col}: {df[col].unique().tolist()}')

Shape: (14, 6)

Class distribution:
play
Yes    9
No     5
Name: count, dtype: int64

Feature unique values:
  outlook: ['Sunny', 'Overcast', 'Rain']
  temp: ['Hot', 'Mild', 'Cool']
  humidity: ['High', 'Normal']
  wind: ['Weak', 'Strong']


---

## Step 1 — The Naive Bayes Formula

Naive Bayes classifies a new example $\mathbf{x} = (x_1, x_2, \ldots, x_n)$ using **Bayes' Theorem**:

$$P(C_k \mid \mathbf{x}) = \frac{P(C_k) \cdot P(\mathbf{x} \mid C_k)}{P(\mathbf{x})}$$

The **"naive" independence assumption** lets us factorise the likelihood:

$$P(\mathbf{x} \mid C_k) = \prod_{i=1}^{n} P(x_i \mid C_k)$$

So the **full scoring formula** (ignoring the constant denominator $P(\mathbf{x})$) is:

$$\boxed{\text{score}(C_k, \mathbf{x}) = P(C_k) \cdot \prod_{i=1}^{n} P(x_i \mid C_k)}$$

The **decision rule** picks the class with the highest score:

$$\hat{y} = \arg\max_{C_k} \; P(C_k) \cdot \prod_{i=1}^{n} P(x_i \mid C_k)$$

### Working in Log-Space

Multiplying many small probabilities causes **numerical underflow** (the product rounds to 0). We convert to **log-space**, turning products into sums:

$$\hat{y} = \arg\max_{C_k} \left[ \log P(C_k) + \sum_{i=1}^{n} \log P(x_i \mid C_k) \right]$$

This is **mathematically identical** to the original formula — $\log$ is monotonically increasing, so the $\arg\max$ does not change.

---

## Step 2 — Prior Probability $P(C_k)$

The **prior** is the probability of each class *before* seeing any features — it is simply the class frequency in the training data:

$$\boxed{P(C_k) = \frac{N_k}{N}}$$

| Symbol | Meaning |
|---|---|
| $N_k$ | Number of training examples labelled $C_k$ |
| $N$ | Total number of training examples |

For the Play Tennis dataset:
- **Yes** appears in 9 of 14 days → $P(\text{Yes}) = 9/14 \approx 0.643$
- **No** appears in 5 of 14 days → $P(\text{No}) = 5/14 \approx 0.357$

In [24]:
# Separate features and target
X = df.drop(columns=['day', 'play'])
y = df['play']

# Compute prior probabilities
N = len(y)                          # total examples
classes = y.unique()                # ['Yes', 'No']
class_counts = y.value_counts()     # N_k for each class

priors = class_counts / N

print('Class counts (N_k):')
print(class_counts.to_string())
print(f'\nTotal examples (N): {N}')
print('\nPrior probabilities P(C_k):')
for cls in classes:
    print(f'  P(play={cls}) = {class_counts[cls]}/{N} = {priors[cls]:.4f}')

Class counts (N_k):
play
Yes    9
No     5

Total examples (N): 14

Prior probabilities P(C_k):
  P(play=No) = 5/14 = 0.3571
  P(play=Yes) = 9/14 = 0.6429


---

## Step 3 — Likelihood $P(x_i = v \mid C_k)$ with Laplace Smoothing

The **likelihood** is the probability of seeing a specific feature value $v$ in class $C_k$.

### Raw (unsmoothed) estimate

$$P(x_i = v \mid C_k) = \frac{\text{count of } (x_i = v \text{ and class} = C_k)}{N_k}$$

**Problem:** If a value $v$ never appears in class $C_k$ in training, this gives $P = 0$, which collapses the entire product to **zero** — no matter how strong the other evidence. This is the **zero-frequency problem**.

### Laplace (Add-$\alpha$) Smoothing

We add a small pseudo-count $\alpha$ to every cell of the frequency table:

$$\boxed{\widehat{P}(x_i = v \mid C_k) = \frac{N_{kiv} + \alpha}{N_k + \alpha \cdot M_i}}$$

| Symbol | Meaning |
|---|---|
| $N_{kiv}$ | Count of training examples in class $C_k$ where feature $i$ equals $v$ |
| $N_k$ | Total training examples in class $C_k$ |
| $\alpha$ | Smoothing parameter ($\alpha = 1$: Laplace smoothing) |
| $M_i$ | Number of distinct values feature $i$ can take |

The denominator adds $\alpha \cdot M_i$ — one $\alpha$ per possible value — so all smoothed probabilities for feature $i$ still **sum to 1**.

In [25]:
# Compute smoothed likelihoods P(x_i = v | C_k)
alpha = 1  # Laplace smoothing

# likelihoods[class][feature][value] = probability
likelihoods = {}

for cls in classes:
    likelihoods[cls] = {}
    X_cls = X[y == cls]               # rows belonging to this class
    N_k   = len(X_cls)                # N_k

    for feature in X.columns:
        M_i = X[feature].nunique()    # number of distinct values
        value_counts = X_cls[feature].value_counts()

        likelihoods[cls][feature] = {}
        for value in X[feature].unique():
            N_kiv = value_counts.get(value, 0)   # count (0 if unseen)
            prob  = (N_kiv + alpha) / (N_k + alpha * M_i)
            likelihoods[cls][feature][value] = prob

# Display the likelihood table for 'outlook'
print('Likelihoods for feature: outlook')
print(f'  (alpha={alpha}, Laplace smoothing)\n')
for cls in classes:
    N_k = class_counts[cls]
    M_i = X['outlook'].nunique()
    print(f'  Class = {cls}  (N_k={N_k}, M_i={M_i})')
    for val, prob in likelihoods[cls]['outlook'].items():
        count = X[y == cls]['outlook'].value_counts().get(val, 0)
        print(f'    P(outlook={val:<8} | play={cls}) = ({count}+{alpha}) / ({N_k}+{alpha}*{M_i}) = {prob:.4f}')
    print()

Likelihoods for feature: outlook
  (alpha=1, Laplace smoothing)

  Class = No  (N_k=5, M_i=3)
    P(outlook=Sunny    | play=No) = (3+1) / (5+1*3) = 0.5000
    P(outlook=Overcast | play=No) = (0+1) / (5+1*3) = 0.1250
    P(outlook=Rain     | play=No) = (2+1) / (5+1*3) = 0.3750

  Class = Yes  (N_k=9, M_i=3)
    P(outlook=Sunny    | play=Yes) = (2+1) / (9+1*3) = 0.2500
    P(outlook=Overcast | play=Yes) = (4+1) / (9+1*3) = 0.4167
    P(outlook=Rain     | play=Yes) = (3+1) / (9+1*3) = 0.3333



In [26]:
# Display ALL likelihood tables as a neat summary
for feature in X.columns:
    print(f'=== P({feature} | play) ===')
    values = sorted(X[feature].unique())
    header = f"{'Value':<12}" + ''.join(f"{cls:<12}" for cls in classes)
    print(header)
    print('-' * (12 + 12 * len(classes)))
    for v in values:
        row = f"{v:<12}" + ''.join(f"{likelihoods[cls][feature][v]:<12.4f}" for cls in classes)
        print(row)
    print()

=== P(outlook | play) ===
Value       No          Yes         
------------------------------------
Overcast    0.1250      0.4167      
Rain        0.3750      0.3333      
Sunny       0.5000      0.2500      

=== P(temp | play) ===
Value       No          Yes         
------------------------------------
Cool        0.2500      0.3333      
Hot         0.3750      0.2500      
Mild        0.3750      0.4167      

=== P(humidity | play) ===
Value       No          Yes         
------------------------------------
High        0.7143      0.3636      
Normal      0.2857      0.6364      

=== P(wind | play) ===
Value       No          Yes         
------------------------------------
Strong      0.5714      0.3636      
Weak        0.4286      0.6364      



---

## Step 4 — The Prediction Algorithm

Given a new example $\mathbf{x} = (x_1, x_2, x_3, x_4)$, the prediction algorithm is:

**For each class $C_k$, compute the log-score:**

$$\text{log-score}(C_k) = \underbrace{\log P(C_k)}_{\text{log-prior}} + \underbrace{\sum_{i=1}^{4} \log P(x_i \mid C_k)}_{\text{sum of log-likelihoods}}$$

**Predict the class with the highest log-score:**

$$\hat{y} = \arg\max_{C_k} \; \text{log-score}(C_k)$$

**Convert log-scores to posterior probabilities** (for interpretability):

$$P(C_k \mid \mathbf{x}) = \frac{e^{\text{log-score}(C_k)}}{\displaystyle\sum_{j} e^{\text{log-score}(C_j)}}$$

This normalised form gives the model's **confidence** in each class.

In [27]:
def predict_proba(sample: dict):
    """
    Returns (log_scores, posteriors) for each class.
    sample: dict mapping feature name -> feature value
    """
    log_scores = {}

    for cls in classes:
        # Start with the log-prior: log P(C_k)
        log_score = np.log(priors[cls])

        # Add log-likelihood for each feature: log P(x_i | C_k)
        for feature, value in sample.items():
            prob = likelihoods[cls][feature].get(
                value,
                alpha / (class_counts[cls] + alpha * X[feature].nunique())
            )
            log_score += np.log(prob)

        log_scores[cls] = log_score

    # Softmax: convert log-scores to probabilities
    max_log = max(log_scores.values())   # subtract max for numerical stability
    exp_scores = {cls: np.exp(s - max_log) for cls, s in log_scores.items()}
    total = sum(exp_scores.values())
    posteriors = {cls: exp_scores[cls] / total for cls in classes}

    return log_scores, posteriors


def predict(sample: dict) -> str:
    """Returns the predicted class label."""
    _, posteriors = predict_proba(sample)
    return max(posteriors, key=posteriors.get)

---

## Step 5 — Worked Prediction Example

Let us classify a new day with these conditions:

| Feature | Value |
|---|---|
| Outlook | Sunny |
| Temperature | Cool |
| Humidity | High |
| Wind | Strong |

We compute the **log-score for each class** step by step:

$$\text{score}(\text{Yes}) = \log P(\text{Yes}) + \log P(\text{Sunny}\mid\text{Yes}) + \log P(\text{Cool}\mid\text{Yes}) + \log P(\text{High}\mid\text{Yes}) + \log P(\text{Strong}\mid\text{Yes})$$

$$\text{score}(\text{No}) = \log P(\text{No}) + \log P(\text{Sunny}\mid\text{No}) + \log P(\text{Cool}\mid\text{No}) + \log P(\text{High}\mid\text{No}) + \log P(\text{Strong}\mid\text{No})$$

The class with the **higher score wins**.

In [28]:
new_day = {
    'outlook' : 'Sunny',
    'temp'    : 'Cool',
    'humidity': 'High',
    'wind'    : 'Strong'
}

log_scores, posteriors = predict_proba(new_day)

print('=== Detailed Log-Score Breakdown ===\n')
for cls in classes:
    running = np.log(priors[cls])
    print(f'Class = {cls}')
    print(f'  log P({cls:<3})                    = log({priors[cls]:.4f}) = {running:.4f}')
    for feature, value in new_day.items():
        p = likelihoods[cls][feature][value]
        running += np.log(p)
        print(f'  log P({feature}={value:<8} | {cls}) = log({p:.4f}) = {np.log(p):.4f}')
    print(f'  {"─"*48}')
    print(f'  Log-score ({cls})                 = {log_scores[cls]:.4f}')
    print()

print('=== Posterior Probabilities (after normalisation) ===')
for cls in classes:
    print(f'  P(play={cls} | X) = {posteriors[cls]:.4f}  ({posteriors[cls]*100:.1f}%)')

prediction = predict(new_day)
print(f'\n>>> Prediction: play = {prediction}')

=== Detailed Log-Score Breakdown ===

Class = No
  log P(No )                    = log(0.3571) = -1.0296
  log P(outlook=Sunny    | No) = log(0.5000) = -0.6931
  log P(temp=Cool     | No) = log(0.2500) = -1.3863
  log P(humidity=High     | No) = log(0.7143) = -0.3365
  log P(wind=Strong   | No) = log(0.5714) = -0.5596
  ────────────────────────────────────────────────
  Log-score (No)                 = -4.0051

Class = Yes
  log P(Yes)                    = log(0.6429) = -0.4418
  log P(outlook=Sunny    | Yes) = log(0.2500) = -1.3863
  log P(temp=Cool     | Yes) = log(0.3333) = -1.0986
  log P(humidity=High     | Yes) = log(0.3636) = -1.0116
  log P(wind=Strong   | Yes) = log(0.3636) = -1.0116
  ────────────────────────────────────────────────
  Log-score (Yes)                 = -4.9499

=== Posterior Probabilities (after normalisation) ===
  P(play=No | X) = 0.7201  (72.0%)
  P(play=Yes | X) = 0.2799  (28.0%)

>>> Prediction: play = No


---

## Step 6 — Evaluate on the Full Training Set

We now run the classifier on all 14 examples and record the prediction alongside the actual label.

> **Note:** Because we evaluate on the same data used for training, this measures **training accuracy** — how well the model fits what it has seen. With only 14 examples a proper train/test split would be too small to be meaningful.

In [29]:
results = []
for _, row in df.iterrows():
    sample    = row[['outlook', 'temp', 'humidity', 'wind']].to_dict()
    actual    = row['play']
    predicted = predict(sample)
    _, post   = predict_proba(sample)
    correct   = predicted == actual
    results.append({
        'Day'      : row['day'],
        'Actual'   : actual,
        'Predicted': predicted,
        'P(Yes)'   : f"{post['Yes']:.3f}",
        'P(No)'    : f"{post['No']:.3f}",
        'Correct'  : '✓' if correct else '✗'
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

accuracy = sum(r['Correct'] == '✓' for r in results) / len(results)
print(f'\nTraining Accuracy: {accuracy:.4f}  ({accuracy*100:.1f}%)')

Day Actual Predicted P(Yes) P(No) Correct
 D1     No        No  0.312 0.688       ✓
 D2     No        No  0.163 0.837       ✓
 D3    Yes       Yes  0.751 0.249       ✓
 D4    Yes       Yes  0.573 0.427       ✓
 D5    Yes       Yes  0.876 0.124       ✓
 D6     No       Yes  0.751 0.249       ✗
 D7    Yes       Yes  0.919 0.081       ✓
 D8     No        No  0.430 0.570       ✓
 D9    Yes       Yes  0.799 0.201       ✓
D10    Yes       Yes  0.855 0.145       ✓
D11    Yes       Yes  0.586 0.414       ✓
D12    Yes       Yes  0.684 0.316       ✓
D13    Yes       Yes  0.930 0.070       ✓
D14     No        No  0.365 0.635       ✓

Training Accuracy: 0.9286  (92.9%)


---

## Step 7 — Verify with scikit-learn `CategoricalNB`

scikit-learn's `CategoricalNB` implements the same Categorical Naive Bayes formula. We compare its predictions against our implementation to verify correctness.

> scikit-learn requires **integer-encoded** features (not strings), so we apply `OrdinalEncoder` first. This encoding is only for API compatibility — it does **not** imply any ordering between categories.

In [30]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

# Encode features as integers (required by CategoricalNB)
enc_X = OrdinalEncoder()
enc_y = LabelEncoder()

X_enc = enc_X.fit_transform(X)
y_enc = enc_y.fit_transform(y)

# Train with same alpha
sk_model = CategoricalNB(alpha=alpha)
sk_model.fit(X_enc, y_enc)

# Predictions
sk_preds_enc = sk_model.predict(X_enc)
sk_preds     = enc_y.inverse_transform(sk_preds_enc)
our_preds    = [
    predict(row[['outlook', 'temp', 'humidity', 'wind']].to_dict())
    for _, row in df.iterrows()
]

# Compare
comparison = pd.DataFrame({
    'Day'    : df['day'].values,
    'Actual' : y.values,
    'Ours'   : our_preds,
    'sklearn': sk_preds,
    'Match'  : ['✓' if a == b else '✗' for a, b in zip(our_preds, sk_preds)]
})

print(comparison.to_string(index=False))
print(f'\nAll predictions match sklearn: {all(comparison["Match"] == "✓")}')
print(f'Our accuracy    : {sum(p == a for p, a in zip(our_preds, y.values)) / len(y):.4f}')
print(f'sklearn accuracy: {sk_model.score(X_enc, y_enc):.4f}')

Day Actual Ours sklearn Match
 D1     No   No      No     ✓
 D2     No   No      No     ✓
 D3    Yes  Yes     Yes     ✓
 D4    Yes  Yes     Yes     ✓
 D5    Yes  Yes     Yes     ✓
 D6     No  Yes     Yes     ✓
 D7    Yes  Yes     Yes     ✓
 D8     No   No      No     ✓
 D9    Yes  Yes     Yes     ✓
D10    Yes  Yes     Yes     ✓
D11    Yes  Yes     Yes     ✓
D12    Yes  Yes     Yes     ✓
D13    Yes  Yes     Yes     ✓
D14     No   No      No     ✓

All predictions match sklearn: True
Our accuracy    : 0.9286
sklearn accuracy: 0.9286


---

## Step 8 — Inspect the Learned Parameters

Let us print the full **likelihood table** — everything our Naive Bayes classifier learned from the 14 training examples.

In [31]:
print(f'Prior Probabilities  (alpha={alpha}):')
print(f'  P(play=Yes) = {priors["Yes"]:.4f}')
print(f'  P(play=No)  = {priors["No"]:.4f}')
print()

for feature in X.columns:
    print(f'Likelihood table: {feature}')
    values = sorted(X[feature].unique())
    print(f"  {'Value':<12} {'P(v|Yes)':>10}  {'P(v|No)':>10}")
    print(f"  {'─'*12} {'─'*10}  {'─'*10}")
    for v in values:
        p_yes = likelihoods['Yes'][feature].get(v, 0)
        p_no  = likelihoods['No'][feature].get(v, 0)
        print(f"  {v:<12} {p_yes:>10.4f}  {p_no:>10.4f}")
    print()

Prior Probabilities  (alpha=1):
  P(play=Yes) = 0.6429
  P(play=No)  = 0.3571

Likelihood table: outlook
  Value          P(v|Yes)     P(v|No)
  ──────────── ──────────  ──────────
  Overcast         0.4167      0.1250
  Rain             0.3333      0.3750
  Sunny            0.2500      0.5000

Likelihood table: temp
  Value          P(v|Yes)     P(v|No)
  ──────────── ──────────  ──────────
  Cool             0.3333      0.2500
  Hot              0.2500      0.3750
  Mild             0.4167      0.3750

Likelihood table: humidity
  Value          P(v|Yes)     P(v|No)
  ──────────── ──────────  ──────────
  High             0.3636      0.7143
  Normal           0.6364      0.2857

Likelihood table: wind
  Value          P(v|Yes)     P(v|No)
  ──────────── ──────────  ──────────
  Strong           0.3636      0.5714
  Weak             0.6364      0.4286



---

## Step 9 — Test with Your Own Custom Input

Change the values in the cell below to any valid combination and run it — the classifier will explain its reasoning step by step.

### Valid values for each feature

| Feature | Allowed values |
|---|---|
| `outlook` | `'Sunny'` · `'Overcast'` · `'Rain'` |
| `temp` | `'Hot'` · `'Mild'` · `'Cool'` |
| `humidity` | `'High'` · `'Normal'` |
| `wind` | `'Weak'` · `'Strong'` |

> **Unseen value?** If you enter a value not seen during training (e.g. `'Foggy'`), Laplace smoothing handles it gracefully — it falls back to $\alpha\,/\,(N_k + \alpha \cdot M_i)$ instead of crashing with zero probability.

In [32]:
# ── CHANGE THESE VALUES TO TEST ──────────────────────────────────────────────
custom_input = {
    'outlook' : 'Rain',       # 'Sunny' | 'Overcast' | 'Rain'
    'temp'    : 'Mild',       # 'Hot'   | 'Mild'     | 'Cool'
    'humidity': 'Normal',     # 'High'  | 'Normal'
    'wind'    : 'Weak',       # 'Weak'  | 'Strong'
}
# ─────────────────────────────────────────────────────────────────────────────

BORDER = '═' * 52

def validate(sample):
    valid = {
        'outlook' : ['Sunny', 'Overcast', 'Rain'],
        'temp'    : ['Hot', 'Mild', 'Cool'],
        'humidity': ['High', 'Normal'],
        'wind'    : ['Weak', 'Strong'],
    }
    warnings = []
    for feat, val in sample.items():
        if val not in valid[feat]:
            warnings.append(f"  ⚠  '{val}' is not a training value for '{feat}' — Laplace fallback used")
    return warnings

warnings = validate(custom_input)

print(BORDER)
print('  YOUR INPUT')
print(BORDER)
for feat, val in custom_input.items():
    print(f'  {feat:<10}: {val}')
if warnings:
    print()
    for w in warnings:
        print(w)
print(BORDER)

════════════════════════════════════════════════════
  YOUR INPUT
════════════════════════════════════════════════════
  outlook   : Rain
  temp      : Mild
  humidity  : Normal
  wind      : Weak
════════════════════════════════════════════════════


In [33]:
log_scores, posteriors = predict_proba(custom_input)
prediction = predict(custom_input)

# ── Step-by-step breakdown ────────────────────────────────────────────────────
print(BORDER)
print('  LOG-SCORE BREAKDOWN')
print(BORDER)

for cls in classes:
    print(f'\n  Class = play={cls}')
    print(f'  {"─"*48}')
    running = np.log(priors[cls])
    print(f'  {"log P(" + cls + ")":<30} = log({priors[cls]:.4f}) = {running:+.4f}')
    for feature, value in custom_input.items():
        p = likelihoods[cls][feature].get(
            value,
            alpha / (class_counts[cls] + alpha * X[feature].nunique())
        )
        contrib = np.log(p)
        running += contrib
        label = f'log P({feature}={value} | {cls})'
        print(f'  {label:<30} = log({p:.4f}) = {contrib:+.4f}')
    print(f'  {"─"*48}')
    print(f'  {"Total log-score":<30} = {log_scores[cls]:.4f}')

# ── Posterior probabilities ───────────────────────────────────────────────────
print(f'\n{BORDER}')
print('  POSTERIOR PROBABILITIES')
print(BORDER)
for cls in classes:
    bar_len = int(posteriors[cls] * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    print(f'  play={cls:<3}  [{bar}]  {posteriors[cls]*100:5.1f}%')

# ── Final verdict ─────────────────────────────────────────────────────────────
print(f'\n{BORDER}')
confidence = posteriors[prediction]
print(f'  PREDICTION  →  play = {prediction}')
print(f'  CONFIDENCE  →  {confidence*100:.1f}%')
print(BORDER)

════════════════════════════════════════════════════
  LOG-SCORE BREAKDOWN
════════════════════════════════════════════════════

  Class = play=No
  ────────────────────────────────────────────────
  log P(No)                      = log(0.3571) = -1.0296
  log P(outlook=Rain | No)       = log(0.3750) = -0.9808
  log P(temp=Mild | No)          = log(0.3750) = -0.9808
  log P(humidity=Normal | No)    = log(0.2857) = -1.2528
  log P(wind=Weak | No)          = log(0.4286) = -0.8473
  ────────────────────────────────────────────────
  Total log-score                = -5.0913

  Class = play=Yes
  ────────────────────────────────────────────────
  log P(Yes)                     = log(0.6429) = -0.4418
  log P(outlook=Rain | Yes)      = log(0.3333) = -1.0986
  log P(temp=Mild | Yes)         = log(0.4167) = -0.8755
  log P(humidity=Normal | Yes)   = log(0.6364) = -0.4520
  log P(wind=Weak | Yes)         = log(0.6364) = -0.4520
  ────────────────────────────────────────────────
  Total log-scor

---

## Summary

### What we built — step by step

| Step | What we computed | Formula |
|---|---|---|
| **Prior** | Class frequency | $P(C_k) = N_k / N$ |
| **Likelihood** | Feature frequency per class, Laplace-smoothed | $\widehat{P}(x_i = v \mid C_k) = (N_{kiv} + \alpha)\,/\,(N_k + \alpha M_i)$ |
| **Log-score** | Unnormalised log-posterior per class | $\log P(C_k) + \sum_i \log P(x_i \mid C_k)$ |
| **Posterior** | Normalised class probabilities | softmax of log-scores |
| **Decision** | Pick highest-scoring class | $\hat{y} = \arg\max_k \; \text{log-score}(C_k)$ |

### Key takeaways

- **Laplace smoothing** ($\alpha = 1$) prevents zero probabilities for unseen feature values.
- **Log-space computation** prevents numerical underflow when multiplying many small probabilities.
- **Categorical NB** is the correct variant for unordered finite categories — not Gaussian (continuous) or Multinomial (counts).
- Our from-scratch implementation **matches scikit-learn's `CategoricalNB` exactly**.

---

### Navigation

| | |
|---|---|
| **Previous** | [Types of Naive Bayes Classifier](Types%20of%20Naive%20Bayes%20Classifier.md) |
| **Detailed Version** | [Simple Code Example — Detailed Each Step](Simple%20Code%20Example%20of%20Naive%20Bayes%20Classifier%20(Detailed%20Each%20Step).ipynb) |
| **Next** | [Naive-Bayes classifier In-Depth](Naive-Bayes%20classifier%20In-Depth.ipynb) |